In [3]:
# Import graph data from preprocessing pipeline into memory
import numpy as np
import pandas as pd

edge_index = np.load("output/processed_fishtree_edge_index.npy")      # graph connectivity
edge_weight = np.load("output/processed_fishtree_edge_weight.npy")    # branch length
nodes = pd.read_csv("output/processed_fishtree_nodes.csv")            # node metadata

In [4]:
# Save graph data into an adjancecy matrix
num_nodes = len(nodes)
A = np.zeros((num_nodes, num_nodes), dtype=np.float32)
A_weighted = np.zeros((num_nodes, num_nodes), dtype=np.float32)
for (u, v), weight in zip(edge_index.T, edge_weight):
    A[u, v] = 1.0
    A_weighted[u, v] = weight
print("Unweighted adjacency matrix:")
print(A)
print("\nWeighted adjacency matrix:")
print(A_weighted)

Unweighted adjacency matrix:
[[0. 1. 0. ... 0. 0. 0.]
 [1. 0. 1. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 1. 1.]
 [0. 0. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 1. 0. 0.]]

Weighted adjacency matrix:
[[0.         0.14494033 0.         ... 0.         0.         0.        ]
 [0.14494033 0.         0.15168737 ... 0.         0.         0.        ]
 [0.         0.15168737 0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.         0.0988199  0.04853752]
 [0.         0.         0.         ... 0.0988199  0.         0.        ]
 [0.         0.         0.         ... 0.04853752 0.         0.        ]]


In [6]:
# Node features: degree, leaf, bias. These are kept simple on purpose, as more features are added from learned embeddings
degree = A.sum(axis=1)
X = np.stack([
    nodes["is_leaf"].astype(np.float32).to_numpy(),
    degree.astype(np.float32),
    nodes["is_leaf"].astype(np.float32).to_numpy(),], axis=1)
feature_names = ["is_leaf", "degree", "leaf_bias"]
X_df = pd.DataFrame(X, columns=feature_names)
X_df.insert(0, "node_name", nodes["node_name"])
X_df

,node_name,is_leaf,degree,leaf_bias
0,internal_0,0.0,2.0,0.0
1,internal_1,0.0,3.0,0.0
2,internal_2,0.0,3.0,0.0
3,internal_3,0.0,3.0,0.0
4,internal_4,0.0,3.0,0.0
...,...,...,...,...
22914,Polypterus retropinnis,1.0,1.0,1.0
22915,Polypterus mokelembembe,1.0,1.0,1.0
22916,internal_22916,0.0,3.0,0.0
22917,Polypterus ornatipinnis,1.0,1.0,1.0


In [7]:
# Normalize adjancecy matrix to avoid bias for highly connected nodes
# We use: A^ = D−1/2(A + I)D−1/2
I = np.eye(num_nodes, dtype=np.float32)
A_tilde = A + I
deg_tilde = A_tilde.sum(axis=1)
D_inv_sqrt = np.diag(1.0 / np.sqrt(deg_tilde))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
np.set_printoptions(precision=3, suppress=True)
print("Normalized adjacency matrix A_hat:")
print(A_hat)

Normalized adjacency matrix A_hat:
[[0.333 0.289 0.    ... 0.    0.    0.   ]
 [0.289 0.25  0.25  ... 0.    0.    0.   ]
 [0.    0.25  0.25  ... 0.    0.    0.   ]
 ...
 [0.    0.    0.    ... 0.25  0.354 0.354]
 [0.    0.    0.    ... 0.354 0.5   0.   ]
 [0.    0.    0.    ... 0.354 0.    0.5  ]]


In [8]:
# Target for training each branch: we use branch length distance [sum of branch lengths between two leaves]
# We need an undirected adjacency list for distance computation (important)
adj = {i: [] for i in range(num_nodes)}

for (u, v), w in zip(edge_index.T, edge_weight):
    u = int(u)
    v = int(v)
    w = float(w)

    adj[u].append((v, w))
    adj[v].append((u, w)) 

In [9]:
# Extract leaf nodes
leaf_mask = nodes["is_leaf"].astype(bool)

leaf_ids = nodes.loc[leaf_mask].index.to_list()
leaf_names = nodes.loc[leaf_mask, "node_name"].to_list()

In [10]:
# Function to calculate distance
def tree_distances_from(start_node: int):
    dist = {start_node: 0.0}
    stack = [start_node]

    while stack:
        u = stack.pop()
        for v, w in adj[u]:
            if v not in dist:
                dist[v] = dist[u] + w
                stack.append(v)

    return dist

In [12]:
# Compute full distance matrix
n_leaves = len(leaf_ids)
leaf_dist = np.zeros((n_leaves, n_leaves), dtype=np.float32)

for i, src in enumerate(leaf_ids):
    d = tree_distances_from(src)

    for j, dst in enumerate(leaf_ids):
        leaf_dist[i, j] = d[dst]

# Normalize distances (to improve stability during training)
max_dist = leaf_dist.max()
if max_dist > 0:
    leaf_dist = leaf_dist / max_dist

# Save to dataframe
leaf_dist_df = pd.DataFrame(
    leaf_dist,
    index=leaf_names,
    columns=leaf_names
)

print(leaf_dist_df)

                         Gambusia geiseri  Gambusia speciosa  \
Gambusia geiseri                 0.000000           0.007801   
Gambusia speciosa                0.007801           0.000000   
Gambusia heterochir              0.008976           0.008435   
Gambusia clarkhubbsi             0.011689           0.011148   
Gambusia krumholzi               0.011351           0.010810   
...                                   ...                ...   
Polypterus ansorgii              0.791913           0.791372   
Polypterus retropinnis           0.782290           0.781749   
Polypterus mokelembembe          0.769163           0.768622   
Polypterus ornatipinnis          0.773323           0.772782   
Polypterus weeksii               0.764472           0.763930   

                         Gambusia heterochir  Gambusia clarkhubbsi  \
Gambusia geiseri                    0.008976              0.011689   
Gambusia speciosa                   0.008435              0.011148   
Gambusia heterochir  